In [2]:
import os
import numpy as np
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

np.random.seed(42)

PROCESSED_DIR = r"C:\Project_EquiGuard\data\processed"
MODELS_DIR    = r"C:\Project_EquiGuard\outputs\models"
ARRAYS_DIR    = os.path.join(PROCESSED_DIR, "arrays")
FIGURES_DIR   = r"C:\Project_EquiGuard\outputs\figures"

os.makedirs(FIGURES_DIR, exist_ok=True)

print("Libraries loaded.")

Libraries loaded.


In [3]:
# Load model and validation data

final_model = joblib.load(os.path.join(MODELS_DIR, "final_model.pkl"))

X_val = np.load(os.path.join(ARRAYS_DIR, "X_val_scaled.npy"))
y_val = np.load(os.path.join(ARRAYS_DIR, "y_val.npy"))

y_prob = final_model.predict_proba(X_val)[:, 1]

print(f"X_val: {X_val.shape}")
print(f"Probabilities computed: {len(y_prob)}")
print(f"Min prob: {y_prob.min():.3f}  Max prob: {y_prob.max():.3f}")

X_val: (132, 29)
Probabilities computed: 132
Min prob: 0.082  Max prob: 1.000


In [4]:
# Evaluate candidate decision thresholds
thresholds = np.arange(0.30, 0.75, 0.05)

precisions = []
recalls    = []
f1_scores  = []

print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12} {'F1':<10}")
print("-" * 46)

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    p = precision_score(y_val, y_pred_t, zero_division=0)
    r = recall_score(y_val, y_pred_t, zero_division=0)
    f = f1_score(y_val, y_pred_t, zero_division=0)
    precisions.append(p)
    recalls.append(r)
    f1_scores.append(f)
    print(f"{t:<12.2f} {p:<12.4f} {r:<12.4f} {f:<10.4f}")

Threshold    Precision    Recall       F1        
----------------------------------------------
0.30         0.9302       0.9836       0.9562    
0.35         0.9297       0.9754       0.9520    
0.40         0.9370       0.9754       0.9558    
0.45         0.9365       0.9672       0.9516    
0.50         0.9435       0.9590       0.9512    
0.55         0.9426       0.9426       0.9426    
0.60         0.9417       0.9262       0.9339    
0.65         0.9496       0.9262       0.9378    
0.70         0.9576       0.9262       0.9417    


In [5]:
#  Select best threshold
# Target: recall >= 0.90 AND precision >= 0.70

best_threshold = None
best_f1        = 0

for i, t in enumerate(thresholds):
    if recalls[i] >= 0.90 and precisions[i] >= 0.70:
        if f1_scores[i] > best_f1:
            best_f1        = f1_scores[i]
            best_threshold = t

if best_threshold is None:
    # Fallback: highest recall if no threshold meets both criteria
    best_threshold = thresholds[np.argmax(recalls)]
    print("No threshold met both criteria — selected highest recall threshold.")
else:
    print(f"Best threshold: {best_threshold:.2f}")
    print(f"F1 at best threshold: {best_f1:.4f}")

print(f"\nAt threshold {best_threshold:.2f}:")
y_pred_best = (y_prob >= best_threshold).astype(int)
print(f"Precision: {precision_score(y_val, y_pred_best):.4f}")
print(f"Recall:    {recall_score(y_val, y_pred_best):.4f}")
print(f"F1:        {f1_score(y_val, y_pred_best):.4f}")

joblib.dump(float(best_threshold), os.path.join(MODELS_DIR, "best_threshold.pkl"))
print(f"\nThreshold saved.")

Best threshold: 0.30
F1 at best threshold: 0.9562

At threshold 0.30:
Precision: 0.9302
Recall:    0.9836
F1:        0.9562

Threshold saved.


In [6]:
# Plot and save

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(thresholds, precisions, marker="o", label="Precision", color="#2196F3")
ax.plot(thresholds, recalls,    marker="s", label="Recall",    color="#4CAF50")
ax.plot(thresholds, f1_scores,  marker="^", label="F1 Score",  color="#FF9800")

ax.axvline(x=best_threshold, color="red", linestyle="--", 
           label=f"Selected threshold = {best_threshold:.2f}")

ax.axhline(y=0.90, color="grey", linestyle=":", alpha=0.7, label="Recall target (0.90)")
ax.axhline(y=0.70, color="grey", linestyle="-.", alpha=0.7, label="Precision target (0.70)")

ax.set_xlabel("Decision Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Precision, Recall and F1 vs Threshold — Validation Set", fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, "precision_recall_threshold.png")
plt.savefig(fig_path, dpi=300)
plt.close()

print("Threshold analysis figure saved.")

Threshold analysis figure saved.
